# sensor_check_stream

**Purpose:** Reconcile one run of `scripts/sensor_simulator.py` (DEC-033) with what Eventstream wrote to Bronze. All
readings are simulated; this is not AVEVA PI System.

**Input:** the newest `Files/raw/sensor/stream_runs/stream_run_<run_id>.json` (the simulator's manifest, uploaded with
`fab cp`) and the Bronze table it names (`bronze_sensor_readings`), filtered to that `run_id`.

**Output tables:**
- `sensor_load_rejects` (append, keyed by `run_id`): rows with a null key or an unknown tag, with the reason
- `pipeline_run_log` (append): one row per check

**Checks:** rows in Bronze = events sent, distinct `(tag_name, ts_utc)` = unique readings, extra rows = deliberate
duplicates, rows at least the threshold behind their timestamp = deliberate late readings, sum of `value` = planned sum,
keys not null, every tag in `sensor_asset_hierarchy`. The notebook raises on any difference. Duplicates and late rows are expected here: Bronze
keeps the feed as it arrived and Silver handles them (stage 3).

Run after a simulator run, a few minutes after the last send so the Lakehouse destination has flushed; a mismatch caused
by running too early clears on a re-run. Not part of a pipeline.


In [ ]:
import glob, json, os
from datetime import datetime
from pyspark.sql import functions as F

manifest_path = max(glob.glob("/lakehouse/default/Files/raw/sensor/stream_runs/stream_run_*.json"))
with open(manifest_path) as fh:
    MANIFEST = json.load(fh)
RUN_ID = MANIFEST["run_id"]
EXPECTED = MANIFEST["expected"]
SOURCE_FILE = os.path.basename(manifest_path)
print("run:", RUN_ID, "| status:", MANIFEST["status"], "| events sent:", MANIFEST["sent_events"])

t0 = datetime.now()
bronze = (spark.table(MANIFEST["bronze_table"]).filter(F.col("run_id") == RUN_ID)
    .withColumn("ts_utc", F.col("ts_utc").cast("timestamp"))
    .withColumn("sent_utc", F.col("sent_utc").cast("timestamp"))
    .withColumn("value", F.col("value").cast("double")))
bronze.cache()

rows = bronze.count()
unique_readings = bronze.select("tag_name", "ts_utc").distinct().count()
lag_seconds = F.unix_timestamp("sent_utc") - F.unix_timestamp("ts_utc")
late_rows = bronze.filter(lag_seconds >= EXPECTED["late_threshold_seconds"]).count()

# Row checks: failed rows go to sensor_load_rejects
tags = spark.table("sensor_asset_hierarchy").filter(F.col("node_type") == "tag").select("tag_name", F.lit(True).alias("_known_tag"))
reason = (F.when(F.col("tag_name").isNull() | F.col("ts_utc").isNull(), "null_key")
           .when(F.col("_known_tag").isNull(), "unknown_tag"))
rejects = (bronze.join(tags, "tag_name", "left").withColumn("reject_reason", reason)
    .filter(F.col("reject_reason").isNotNull())
    .select("tag_name", "ts_utc", "value", "source", "reject_reason",
            F.lit(SOURCE_FILE).alias("_source_file"), F.lit(RUN_ID).alias("run_id")))
n_rejects = rejects.count()

checks = [
    ("manifest status", MANIFEST["status"], "COMPLETED"),
    ("rows = events sent", rows, MANIFEST["sent_events"]),
    ("events sent = planned", MANIFEST["sent_events"], EXPECTED["events"]),
    ("distinct keys = unique readings", unique_readings, EXPECTED["unique_readings"]),
    ("extra rows = duplicates", rows - unique_readings, EXPECTED["duplicates"]),
    ("late rows", late_rows, EXPECTED["late"]),
    ("value sum", round(bronze.agg(F.sum("value")).first()[0] or 0, 3), EXPECTED["value_sum"]),
    ("rejected rows", n_rejects, 0),
]
for name, actual, expected in checks:
    print(f"{'OK  ' if actual == expected else 'FAIL'} {name}: {actual} (expected {expected})")
print("timestamps:", bronze.agg(F.min("ts_utc"), F.max("ts_utc")).first())
print("columns:", bronze.columns)
failed = [name for name, actual, expected in checks if actual != expected]

if n_rejects:
    rejects.write.format("delta").mode("append").saveAsTable("sensor_load_rejects")
LOG_SCHEMA = ("run_id string, layer string, source string, file string, rows_in long, rows_out long, "
              "status string, message string, started_at timestamp, ended_at timestamp")
log_row = (RUN_ID, "bronze", "sensor_stream", SOURCE_FILE, MANIFEST["sent_events"], rows,
           "FAILED" if failed else "SUCCESS", str(failed)[:500] if failed else "", t0, datetime.now())
spark.createDataFrame([log_row], LOG_SCHEMA).write.format("delta").mode("append").saveAsTable("pipeline_run_log")

if failed:
    raise ValueError(f"sensor stream {RUN_ID} does not reconcile: {failed}")
print("Reconciled:", rows, "rows in Bronze for run", RUN_ID)
